In [1]:
import os 
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    '../artifacts/data/processed/train',
    label_mode='int',
    image_size=(224, 224),
    shuffle=True
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    '../artifacts/data/processed/val',
    label_mode='int',
    image_size=(224, 224),
    shuffle=True
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    '../artifacts/data/processed/test',
    label_mode='int',
    image_size=(224, 224),
    shuffle=False
)

Found 10363 files belonging to 2 classes.
Found 3157 files belonging to 2 classes.
Found 3138 files belonging to 2 classes.


In [3]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.1)
])


In [4]:
base_model = tf.keras.applications.MobileNetV2(
            weights='imagenet',
            input_shape=(224, 224, 3),
            include_top=False
        )

In [ ]:
base_model.trainable = False

num_classes = 2


model = models.Sequential([
    layers.InputLayer(shape=(224, 224, 3)),
    data_augmentation,
    layers.Lambda(tf.keras.applications.mobilenet_v2.preprocess_input),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(64, activation='relu'),
    layers.Dense(num_classes, activation='softmax')
])

d:\SAMITH\Github\Image-Based-Food-Freshness-Prediction-System\venv\Lib\site-packages\keras\src\layers\core\input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


In [6]:
model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
            loss='sparse_categorical_crossentropy',
            metrics=["accuracy"]
        )

In [7]:
early_stopping = EarlyStopping(
            monitor='val_loss',    
            patience=5,
            restore_best_weights=True
        )

In [8]:
history = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=50,
            callbacks=[early_stopping]  
        )

Epoch 1/50
324/324 ━━━━━━━━━━━━━━━━━━━━ 425s 1s/step - accuracy: 0.9134 - loss: 0.2115 - val_accuracy: 0.9566 - val_loss: 0.1159
Epoch 2/50
324/324 ━━━━━━━━━━━━━━━━━━━━ 483s 1s/step - accuracy: 0.9559 - loss: 0.1166 - val_accuracy: 0.9636 - val_loss: 0.1040
Epoch 3/50
324/324 ━━━━━━━━━━━━━━━━━━━━ 455s 1s/step - accuracy: 0.9622 - loss: 0.0964 - val_accuracy: 0.9743 - val_loss: 0.0757
Epoch 4/50
324/324 ━━━━━━━━━━━━━━━━━━━━ 452s 1s/step - accuracy: 0.9718 - loss: 0.0752 - val_accuracy: 0.9690 - val_loss: 0.0858
Epoch 5/50
324/324 ━━━━━━━━━━━━━━━━━━━━ 442s 1s/step - accuracy: 0.9698 - loss: 0.0759 - val_accuracy: 0.9753 - val_loss: 0.0752
Epoch 6/50
324/324 ━━━━━━━━━━━━━━━━━━━━ 421s 1s/step - accuracy: 0.9793 - loss: 0.0596 - val_accuracy: 0.9807 - val_loss: 0.0560
Epoch 7/50
324/324 ━━━━━━━━━━━━━━━━━━━━ 466s 1s/step - accuracy: 0.9783 - loss: 0.0558 - val_accuracy: 0.9823 - val_loss: 0.0512
Epoch 8/50
324/324 ━━━━━━━━━━━━━━━━━━━━ 436s 1s/step - accuracy: 0.9812 - loss: 0.0497 - val_accu

In [9]:

os.makedirs('artifacts/models', exist_ok=True) 
model.save('artifacts/models/with_augmentation.keras')


In [10]:
from keras.callbacks import ReduceLROnPlateau

In [15]:
base_model.trainable = False

for layer in base_model.layers[:-20]:  
    layer.trainable = False

num_classes = 2


model = models.Sequential([
    layers.InputLayer(shape=(224, 224, 3)),
    data_augmentation,
    layers.Lambda(tf.keras.applications.mobilenet_v2.preprocess_input),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(64, activation='relu'),
    layers.Dense(num_classes, activation='softmax')
])

In [16]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

In [17]:
early_stopping = EarlyStopping(
            monitor='val_loss',    
            patience=5,
            restore_best_weights=True
        )

In [18]:
lr_schedule = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7)


history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=[early_stopping, lr_schedule]
)

Epoch 1/50
324/324 ━━━━━━━━━━━━━━━━━━━━ 533s 2s/step - accuracy: 0.6353 - loss: 0.6401 - val_accuracy: 0.7532 - val_loss: 0.4958 - learning_rate: 1.0000e-05
Epoch 2/50
324/324 ━━━━━━━━━━━━━━━━━━━━ 539s 2s/step - accuracy: 0.7768 - loss: 0.4643 - val_accuracy: 0.8207 - val_loss: 0.3949 - learning_rate: 1.0000e-05
Epoch 3/50
324/324 ━━━━━━━━━━━━━━━━━━━━ 479s 1s/step - accuracy: 0.8338 - loss: 0.3835 - val_accuracy: 0.8508 - val_loss: 0.3403 - learning_rate: 1.0000e-05
Epoch 4/50
324/324 ━━━━━━━━━━━━━━━━━━━━ 418s 1s/step - accuracy: 0.8557 - loss: 0.3365 - val_accuracy: 0.8742 - val_loss: 0.3037 - learning_rate: 1.0000e-05
Epoch 5/50
324/324 ━━━━━━━━━━━━━━━━━━━━ 473s 1s/step - accuracy: 0.8760 - loss: 0.3007 - val_accuracy: 0.8879 - val_loss: 0.2763 - learning_rate: 1.0000e-05
Epoch 6/50
324/324 ━━━━━━━━━━━━━━━━━━━━ 444s 1s/step - accuracy: 0.8897 - loss: 0.2766 - val_accuracy: 0.8967 - val_loss: 0.2548 - learning_rate: 1.0000e-05
Epoch 7/50
324/324 ━━━━━━━━━━━━━━━━━━━━ 440s 1s/step - acc